# tensor-item-scalar — ex9: .item() breaks autograd — diagnose the silent graph severing

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-item-scalar`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-item-scalar`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-item-scalar"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## .item() autograd pitfall — quick refresher

`.item()` returns a **Python float / int / bool** — a primitive with no autograd metadata. Any expression that funnels a `requires_grad=True` tensor through `.item()` becomes a constant downstream: the gradient graph is **silently severed**.

```python
loss = ((y_pred - y_true) ** 2).mean()
scalar = loss.item()             # <- breaks the graph
really_loss = scalar * weight     # <- weight.grad will be None
```

**Compared to `.detach()`.** `.detach()` returns a tensor that's disconnected from the graph but still a tensor — so an obvious type mismatch downstream will catch the bug. `.item()` returns a primitive, which silently quacks like a number and only manifests later as `.grad is None`.

**This drill (ex9) vs ex1-8.** Earlier exercises use `.item()` for its intended purpose — extracting scalars for logging and Python-side control flow. ex9 deliberately misuses `.item()` inside an autograd path and forces the caller to inspect `.grad` to diagnose the silent breakage.

### Exercise 9 — .item() breaks autograd — diagnose the silent graph severing

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze a forward pass that mixes a correct (tensor-only) loss path with an incorrect (`.item()`-laundered) loss path, and produce a diagnostic showing the correct path has a non-None gradient while the incorrect path does not.
> Keywords: autograd-pitfall, graph-severing, diagnostic
> ```

**KCs targeted:** `item-returns-python-primitive`, `item-breaks-autograd-graph`

Implement `ex9_diagnose_item_breakage(x_init, target)`.

Build TWO parallel loss computations from the same inputs and report which one preserves gradient flow back to `x`:

1. `x_init` is a tensor `(N,)` of starting values (will be wrapped fresh with `requires_grad=True` inside each path). `target` is `(N,)`, no grad.
2. **Correct path.** `x_ok = x_init.clone().detach().requires_grad_(True)`. Compute `loss_ok = ((x_ok - target) ** 2).mean()` (pure-tensor). Call `loss_ok.backward()`. Record `x_ok.grad.clone()`.
3. **Broken path.** `x_bad = x_init.clone().detach().requires_grad_(True)`. Compute `diff_scalar = (x_bad - target).pow(2).mean().item()` — note the `.item()`. Then build `loss_bad = t.tensor(diff_scalar)` (a fresh Python-float-derived tensor with no graph back to `x_bad`). Wrap `loss_bad.backward()` in `try/except` and continue regardless. `x_bad.grad` will remain `None`.
4. **Return** a dict:
   - `'x_ok_grad_norm'`: float — `x_ok.grad.norm().item()` (>0)
   - `'x_bad_grad'`: tensor or `None` — `x_bad.grad` itself
   - `'graph_preserved'`: bool — `x_ok.grad is not None and x_bad.grad is None`

**Print** the dict so the diagnostic is visible to the caller without rerunning.

Output: dict with the three keys above. The visualization renders a bar chart comparing the two paths' gradient norms — the broken path is forced to 0 because `x_bad.grad` is `None`.

In [ ]:
def ex9_diagnose_item_breakage(x_init: Tensor, target: Tensor) -> dict:
    # Correct path — pure tensor arithmetic. Gradient flows back to x_ok.
    x_ok = x_init.clone().detach().requires_grad_(True)
    loss_ok = ((x_ok - target) ** 2).mean()
    loss_ok.backward()

    # Broken path — .item() returns a Python float; loss_bad is built from
    # that float wrapped in a fresh tensor, so there's no graph link back to x_bad.
    x_bad = x_init.clone().detach().requires_grad_(True)
    diff_scalar = (x_bad - target).pow(2).mean().item()
    loss_bad = t.tensor(diff_scalar)
    try:
        loss_bad.backward()
    except RuntimeError:
        # Expected — loss_bad has no grad_fn because it was built from a Python float.
        pass

    out = {
        'x_ok_grad_norm': x_ok.grad.norm().item() if x_ok.grad is not None else None,
        'x_bad_grad': x_bad.grad,
        'graph_preserved': (x_ok.grad is not None) and (x_bad.grad is None),
    }
    print(out)
    return out


<details><summary>Solution</summary>

```python
def ex9_diagnose_item_breakage(x_init: Tensor, target: Tensor) -> dict:
    # Correct path — pure tensor arithmetic. Gradient flows back to x_ok.
    x_ok = x_init.clone().detach().requires_grad_(True)
    loss_ok = ((x_ok - target) ** 2).mean()
    loss_ok.backward()

    # Broken path — .item() returns a Python float; loss_bad is built from
    # that float wrapped in a fresh tensor, so there's no graph link back to x_bad.
    x_bad = x_init.clone().detach().requires_grad_(True)
    diff_scalar = (x_bad - target).pow(2).mean().item()
    loss_bad = t.tensor(diff_scalar)
    try:
        loss_bad.backward()
    except RuntimeError:
        # Expected — loss_bad has no grad_fn because it was built from a Python float.
        pass

    out = {
        'x_ok_grad_norm': x_ok.grad.norm().item() if x_ok.grad is not None else None,
        'x_bad_grad': x_bad.grad,
        'graph_preserved': (x_ok.grad is not None) and (x_bad.grad is None),
    }
    print(out)
    return out
```

**Why w_ok.grad equals the MSE.** `loss_ok = w_ok · MSE(x, target)` is linear in `w_ok`, so `∂loss_ok / ∂w_ok = MSE(x, target)`. Autograd fills `w_ok.grad` with this scalar after `backward()`.

**Why w_bad.grad is None.** `mse_scalar = ....item()` returned a Python float. The product `w_bad * mse_scalar` is a tensor-times-constant — autograd treats `mse_scalar` as a non-differentiable literal, so backward never even tries to flow gradient through it. Worse, in some torch versions `loss_bad.backward()` may raise because there's nothing to differentiate from `w_bad` to a meaningful upstream — hence the try/except.

**Defensive habits.** (1) Never use `.item()` inside a forward pass that you intend to differentiate. (2) If you need a graph-detached tensor (e.g. for logging or saving), use `.detach()` not `.item()` — the type mismatch will surface downstream. (3) After `backward()`, always sanity-check a small model with `assert all(p.grad is not None for p in model.parameters())` until you trust the data path.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()